# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Show metadata
md = dataset.metadata  # This is a Metadata object
print(f"Dataset name: {md.name}")
print(f"Description: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` references. Each entity in the Croissant schema is uniquely identified by its `@id`.

Below, we display the record sets and their structure using the dataset's metadata.

In [ ]:
# List all record set `@id`s in the dataset
record_sets = dataset.metadata.record_sets
if not record_sets:
    print('No record sets found in this dataset.')
else:
    print('Record Sets in Dataset:')
    for rs in record_sets:
        print(f"@id: {rs.id} | name: {rs.name}")
        # Show fields/columns in the record set
        print('  Fields:')
        for field in rs.fields:
            print(f"    @id: {field.id} | name: {field.name} | dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

We'll use `@id` references for the record sets and fields.

In [ ]:
# Find all record set @ids
record_sets = dataset.metadata.record_sets
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Collect records from this record set
    print(f"Extracting records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"  No records found for {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Columns: {df.columns.tolist()}")
    display(df.head())

# For demonstration, pick the first record set (if exists)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Using main_record_set_id for EDA: {main_record_set_id}")
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter numeric fields, normalize, and group. We use `@id` to refer to the fields/columns.

**Note:** Be sure to select valid `@id` values for numeric and grouping fields from the overview above.

In [ ]:
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]

    # List all columns. Pick a numeric field @id for the demo.
    # We try to auto-detect the first numeric column.
    numeric_field_id = None
    for rs in record_sets:
        if rs.id == main_record_set_id:
            for f in rs.fields:
                # Accept Integer/Float/Number
                if f.data_type in ('schema:Integer', 'schema:Float', 'schema:Number') and f.id in df.columns:
                    numeric_field_id = f.id
                    break
    if numeric_field_id:
        print(f"Using numeric field for EDA: {numeric_field_id}")
        # Convert column to float in case it's object
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        # Filter: keep values greater than the median
        threshold = df[numeric_field_id].median()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        display(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
            filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to auto-select a grouping field (categorical/label)
        group_field_id = None
        for rs in record_sets:
            if rs.id == main_record_set_id:
                for f in rs.fields:
                    if (f.data_type == 'schema:Text' or f.data_type is None) and f.id in df.columns:
                        group_field_id = f.id
                        break
        if group_field_id:
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No categorical field found for grouping.")
    else:
        print("No numeric field found for EDA in this record set.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and numeric_field_id:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field exists, make a boxplot
    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion
In this notebook, we:
* Loaded the Croissant dataset metadata and records using their `@id` with `mlcroissant`.
* Reviewed record set and field structure by `@id`.
* Extracted and loaded records into Pandas DataFrames.
* Performed filtering, normalization, and grouping using `@id` identifiers.
* Visualized numeric fields and grouped statistics.

This modular notebook demonstrates how to process complex FAIR datasets with `mlcroissant` by referencing all elements by their unique Croissant `@id`.